# Throttle quickstart: your real $ per million tokens on a free GPU

This notebook serves a small model on a free **T4 GPU**, measures what it costs per million output tokens with
[Throttle](https://github.com/KushagraKanaujia/throttle), changes one serving setting, and asks Throttle whether the
change made tokens cheaper, more expensive, or made no difference beyond run-to-run noise.

**Before you start**
- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- **Kaggle:** Settings → Accelerator → **GPU T4**, and turn **Internet** on.

Then run the cells top to bottom (about 5–10 minutes, mostly the model download and the repeat checks).

The GPU price is an **ASSUMED** $0.35/hr, a typical rental price for a T4; the notebook itself is free. Tokens and time
are **MEASURED**.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Pick an engine

- `ollama` (default): installs in about a minute and works reliably on a free T4.
- `vllm`: the engine most production teams use; installing it on Colab can take several minutes.

If the vLLM install fails, set `ENGINE = "ollama"` and re-run from here.

In [ ]:
ENGINE = "ollama"          # or "vllm"
GPU_RATE = 0.35            # $/hr, ASSUMED (typical T4 rental price)
OLLAMA_MODEL = "qwen2.5:0.5b"
VLLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# Same small workload for every check, so checks are comparable.
WORKLOAD = "--blocks 3 --requests-per-block 8 --concurrency 4 --max-tokens 128"

## 3. Install the engine and Throttle

In [ ]:
# Untested here: needs a Colab/Kaggle GPU runtime.
import subprocess
if ENGINE == "ollama":
    subprocess.run("apt-get -qq install -y zstd > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, check=True)
else:
    subprocess.run("pip install -q vllm", shell=True, check=True)
subprocess.run("pip install -q throttle-pro==0.4.2", shell=True, check=True)
!throttle --version

## 4. Start the server

`start_server(setting)` (re)starts the engine with one serving setting:
- **Ollama:** `OLLAMA_NUM_PARALLEL`, how many requests it serves at once.
- **vLLM:** `--max-num-seqs`, the same idea.

In [ ]:
# Untested here: needs a Colab/Kaggle GPU runtime.
import os, subprocess, time, urllib.request

proc = None

def start_server(setting):
    """(Re)start the engine with one setting: parallel requests (Ollama) or max-num-seqs (vLLM)."""
    global proc, URL, MODEL
    subprocess.run("pkill -f 'ollama serve' ; pkill -f 'vllm serve'", shell=True)
    time.sleep(3)
    if ENGINE == "ollama":
        env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(setting), OLLAMA_HOST="127.0.0.1:11434")
        proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
        URL, MODEL, probe = "http://127.0.0.1:11434", OLLAMA_MODEL, "http://127.0.0.1:11434/api/tags"
    else:
        proc = subprocess.Popen(
            ["vllm", "serve", VLLM_MODEL, "--dtype", "half", "--max-model-len", "4096",
             "--gpu-memory-utilization", "0.85", "--max-num-seqs", str(setting), "--port", "8000"],
            stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
        URL, MODEL, probe = "http://127.0.0.1:8000", VLLM_MODEL, "http://127.0.0.1:8000/v1/models"
    for _ in range(300):  # up to ~10 minutes (first vLLM start downloads the model)
        try:
            urllib.request.urlopen(probe, timeout=2)
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError("server did not start; see server.log")
    if ENGINE == "ollama":
        subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True, stdout=subprocess.DEVNULL)
    print(f"{ENGINE} is up at {URL}, serving {MODEL}, setting = {setting}")

BASELINE, CHANGED = 1, 4   # parallel requests / max-num-seqs: before and after
start_server(BASELINE)

## 5. Measure the baseline four times

One check gives a cost with a 95% interval, but two runs at different times differ from machine load alone.
Throttle only calls a winner after **repeat checks of the same config** have measured that run-to-run noise.
The first check says it's the first, the next ones say NOT CALIBRATED or NO WINNER: that's expected.

In [ ]:
# Untested here: needs a Colab/Kaggle GPU runtime.
for i in range(4):
    print(f"\n===== baseline check {i + 1} of 4 =====")
    !throttle check --url {URL} --model {MODEL} --gpu-hourly-rate {GPU_RATE} --label parallel-{BASELINE} --config gpu=T4 --config parallel={BASELINE} {WORKLOAD}

## 6. Change one setting, then check again

Now serve up to **4 requests at once** instead of 1 and run the same check. Throttle compares it with the baseline
and prints a verdict: **CHEAPER**, **MORE EXPENSIVE**, or **NO WINNER** if the difference is inside the noise.

In [ ]:
# Untested here: needs a Colab/Kaggle GPU runtime.
start_server(CHANGED)
!throttle check --url {URL} --model {MODEL} --gpu-hourly-rate {GPU_RATE} --label parallel-{CHANGED} --config gpu=T4 --config parallel={CHANGED} {WORKLOAD}

## 7. Your history

In [ ]:
!throttle check --history

## 8. Share your result

This prints a markdown summary (no URLs, hostnames, IPs or keys) and a link to a pre-filled GitHub issue.
Nothing is uploaded unless you open the link and submit it. Every shared result helps build an open picture of what
serving changes actually cost.

In [ ]:
!throttle check --history --share

## What next

- Point Throttle at **your own** vLLM, SGLang or Ollama server: `pipx install throttle-pro`, then
  `throttle check --url http://your-server:8000 --gpu-hourly-rate <your $/hr>`.
- Gate deploys on cost with the GitHub Action: https://github.com/KushagraKanaujia/throttle/blob/main/docs/github-action.md
- Want us to measure your stack for you? https://throttle-pro.com/audit